In [4]:
import numpy as np

In [5]:

def np_quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
    """Multiply quaternions in NumPy, supports (N, 4) and (4,) inputs."""
    q1_was_1d = q1.ndim == 1
    q2_was_1d = q2.ndim == 1

    if q1_was_1d:
        q1 = q1[None, :]
    if q2_was_1d:
        q2 = q2[None, :]

    if q1.shape[0] == 1 and q2.shape[0] > 1:
        q1 = np.broadcast_to(q1, q2.shape)
    elif q2.shape[0] == 1 and q1.shape[0] > 1:
        q2 = np.broadcast_to(q2, q1.shape)

    w1, x1, y1, z1 = q1[:, 0], q1[:, 1], q1[:, 2], q1[:, 3]
    w2, x2, y2, z2 = q2[:, 0], q2[:, 1], q2[:, 2], q2[:, 3]
    result = np.stack(
        [
            w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
            w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
            w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
            w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2,
        ],
        axis=1,
    )
    return result[0] if q1_was_1d and q2_was_1d else result


In [6]:
# 1. 单个四元数相乘
q_id = np.array([1.0, 0.0, 0.0, 0.0])  # 单位四元数（无旋转）
q_rot = np.array([0.0, 1.0, 0.0, 0.0])
res1 = np_quat_mul(q_id, q_rot)
print("【单个四元数】结果:", res1, "形状:", res1.shape)

# 2. 批量四元数相乘
batch_q1 = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
batch_q2 = np.array([[0, 0, 1, 0], [1, 0, 0, 0]])
res2 = np_quat_mul(batch_q1, batch_q2)
print("\n【批量四元数】结果:\n", res2, "\n形状:", res2.shape)

# 3. 单个 × 批量（广播测试）
single_q = np.array([1, 0, 0, 0])
batch_q = np.array([[0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
res3 = np_quat_mul(single_q, batch_q)
print("\n【广播运算】结果:\n", res3, "\n形状:", res3.shape)


【单个四元数】结果: [0. 1. 0. 0.] 形状: (4,)

【批量四元数】结果:
 [[0 0 1 0]
 [0 1 0 0]] 
形状: (2, 4)

【广播运算】结果:
 [[0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]] 
形状: (3, 4)


In [7]:
import numpy as np

# 原数组：1行4列（单个四元数）
a = np.array([[1, 2, 3, 4]])
print("原形状:", a.shape)  # (1, 4)

# 广播拉伸为 3行4列
b = np.broadcast_to(a, (3, 4))
print("\n拉伸后:\n", b)
print("拉伸后形状:", b.shape)  # (3, 4)


原形状: (1, 4)

拉伸后:
 [[1 2 3 4]
 [1 2 3 4]
 [1 2 3 4]]
拉伸后形状: (3, 4)


In [8]:
import numpy as np
# 假设有 2 组结果
w = np.array([1, 2])
x = np.array([3, 4])
y = np.array([5, 6])
z = np.array([7, 8])

res = np.stack([w, x, y, z], axis=1)
print(res)


[[1 3 5 7]
 [2 4 6 8]]


In [19]:
import numpy as np

# 构造 3 个形状完全一致的数组：shape = (3, 3)
arr1 = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

arr2 = np.array([
    [10, 11, 12],
    [13, 14, 15],
    [16, 17, 18]
])

arr3 = np.array([
    [19, 20, 21],
    [22, 23, 24],
    [25, 26, 27]
])


print("单个数组形状:", arr1.shape)  # 输出 (3, 3)
print("=" * 40)

# 1. 重点：axis=-1 最后一维拼接 → 得到 (3, 3, 3)
res_last = np.stack([arr1, arr2, arr3], axis=-1)
print("axis=-1 结果（形状 3×3×3）：")
print(res_last)
print("输出形状:", res_last.shape)  # (3, 3, 3)
print("=" * 40)

# 2. 对比：axis=0 最前维度拼接
res_0 = np.stack([arr1, arr2, arr3], axis=0)
print(res_0)
print("axis=0 结果形状:", res_0.shape)  # (3, 3, 3)，排布逻辑不同
print("=" * 40)

# 3. 对比：axis=1 中间维度拼接
res_1 = np.stack([arr1, arr2, arr3], axis=1)
print(res_1)
print("axis=1 结果形状:", res_1.shape)  # (3, 3, 3)


[19 20 21]
[22 23 24]
[25 26 27]
单个数组形状: (3, 3)
axis=-1 结果（形状 3×3×3）：
[[[ 1 10 19]
  [ 2 11 20]
  [ 3 12 21]]

 [[ 4 13 22]
  [ 5 14 23]
  [ 6 15 24]]

 [[ 7 16 25]
  [ 8 17 26]
  [ 9 18 27]]]
输出形状: (3, 3, 3)
[[[ 1  2  3]
  [ 4  5  6]
  [ 7  8  9]]

 [[10 11 12]
  [13 14 15]
  [16 17 18]]

 [[19 20 21]
  [22 23 24]
  [25 26 27]]]
axis=0 结果形状: (3, 3, 3)
[[[ 1  2  3]
  [10 11 12]
  [19 20 21]]

 [[ 4  5  6]
  [13 14 15]
  [22 23 24]]

 [[ 7  8  9]
  [16 17 18]
  [25 26 27]]]
axis=1 结果形状: (3, 3, 3)


In [20]:
import torch

# 查看CUDA是否可用（True = GPU版，False = 仅CPU）
print(torch.cuda.is_available())

True


In [21]:
print(torch.cuda.device_count())

1


In [22]:
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 5060 Ti
